# 02 · MMDetection models — leakage-safe V5

Run top to bottom. Existing Honours_Benchmark results are untouched. Start with the two-epoch smoke checks. Long training remains locked until they succeed.

Training and validation curves use separate splits. Test images never control patience or checkpoint selection.

## 1. Mount Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import sys
import torch

print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Select the G4 GPU runtime first")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
    "GB",
)

!nvidia-smi

Mounted at /content/drive
Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
CUDA available: True
GPU: Tesla T4
GPU memory: 14.6 GB
Mon Sep  7 14:04:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |


In [3]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")
ROOT = DRIVE / "SAR_Ship_Segmentation_Study"

assert ROOT.is_dir(), f"Project not found: {ROOT}"

print("Project:", ROOT)

Project: /content/drive/MyDrive/SAR_Ship_Segmentation_Study


In [ ]:
ARCHITECTURES = (
    ROOT
    / "sarstudy"
    / "reference"
    / "models"
    / "architectures.py"
)

assert ARCHITECTURES.is_file(), f"Missing: {ARCHITECTURES}"

architecture_code = ARCHITECTURES.read_text()

assert "def paper_training_config" in architecture_code, (
    "The MMDetection import correction is missing"
)

print("Architecture correction is present")

NameError: name 'ROOT' is not defined

In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        str(ROOT / "scripts" / "setup_environment.py"),
        "mmdetection",
    ],
    check=True,
)

PYTHON = Path("/content/sar-mmdetection-env/bin/python")

assert PYTHON.is_file(), f"Environment was not created: {PYTHON}"

print("Environment ready:", PYTHON)

Environment ready: /content/sar-mmdetection-env/bin/python


In [4]:
subprocess.run(
    [
        str(PYTHON),
        "-c",
        """
import torch
import mmengine
import mmcv
import mmdet

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("MMEngine:", mmengine.__version__)
print("MMCV:", mmcv.__version__)
print("MMDetection:", mmdet.__version__)

from sarstudy.reference.models.architectures import paper_training_config
print("Architecture registry:", paper_training_config("swininsseg", "ssdd"))
""",
    ],
    check=True,
)

CompletedProcess(args=['/content/sar-mmdetection-env/bin/python', '-c', '\nimport torch\nimport mmengine\nimport mmcv\nimport mmdet\n\nprint("Torch:", torch.__version__)\nprint("CUDA:", torch.cuda.is_available())\nprint("GPU:", torch.cuda.get_device_name(0))\nprint("MMEngine:", mmengine.__version__)\nprint("MMCV:", mmcv.__version__)\nprint("MMDetection:", mmdet.__version__)\n\nfrom sarstudy.reference.models.architectures import paper_training_config\nprint("Architecture registry:", paper_training_config("swininsseg", "ssdd"))\n'], returncode=0)

## 2. Install packages

Run once per session and wait for READY. This installs packages, not training. No restart is needed. If installation fails, stop here and copy the error above the final traceback; do not run the training cells.

Use an A100 (Python 3.11) for this notebook. Do not use Blackwell with this legacy stack.

## 3. ClearML credentials

In [7]:
from google.colab import userdata
import os
# Secrets stay in this session's environment; they are never written into the package.
os.environ['CLEARML_API_ACCESS_KEY'] = userdata.get('CLEARML_ACCESS_KEY')
os.environ['CLEARML_API_SECRET_KEY'] = userdata.get('CLEARML_SECRET_KEY')
os.environ['CLEARML_API_HOST'] = 'https://api.clear.ml'
os.environ['CLEARML_WEB_HOST'] = 'https://app.clear.ml'
os.environ['CLEARML_FILES_HOST'] = 'https://files.clear.ml'
print('ClearML credentials configured')


ClearML credentials configured


## 4. Frozen study settings

In [ ]:
import json

BASE_CONFIG = ROOT / "configs" / "study.json"
FINAL_CONFIG = ROOT / "configs" / "mmdetection_v9_fresh.json"

settings = json.loads(BASE_CONFIG.read_text())

settings["experiment"] = "study_v9_swininsseg_fresh"
settings["models"] = ["swininsseg"]
settings["datasets"] = ["ssdd", "hrsid"]

settings["batch_size"] = 1
settings["workers"] = 0
settings["amp"] = True

settings["max_epochs"] = 500
settings["patience"] = 20
settings["seed"] = 42

settings["clearml"] = True

FINAL_CONFIG.write_text(json.dumps(settings, indent=2))
CONFIG = FINAL_CONFIG

print("Experiment:", settings["experiment"])
print("Models:", settings["models"])
print("Datasets:", settings["datasets"])
print("Maximum epochs:", settings["max_epochs"])
print("Patience:", settings["patience"])
print("Batch size:", settings["batch_size"])
print("Workers:", settings["workers"])
print("ClearML:", settings["clearml"])

Experiment: study_v9_swininsseg_fresh
Models: ['swininsseg']
Datasets: ['ssdd', 'hrsid']
Maximum epochs: 500
Patience: 20
Batch size: 1
Workers: 0
ClearML: True


## Run helpers

In [ ]:
import os
import subprocess

def run_one(model, dataset, smoke=True):
    command = [
        str(PYTHON),
        "-u",
        "-m",
        "sarstudy.runner",
        "--config",
        str(CONFIG),
        "--model",
        model,
        "--dataset",
        dataset,
    ]

    if smoke:
        command.append("--smoke")

    environment = os.environ.copy()
    environment["MPLBACKEND"] = "Agg"

    print("Running:", model, dataset, "| smoke:", smoke)

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=environment,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"{model}/{dataset} failed with exit code {return_code}"
        )

    print(f"\n✅ Completed: {model}/{dataset}")

## Two-epoch verification

Run one cell at a time. Check that train_loss, val_loss and validation AP appear and that ClearML has the Loss / total graph. A smoke pass checks execution, not convergence.

In [ ]:
for dataset in ["ssdd", "hrsid"]:
    print("\n" + "=" * 70)
    print("SMOKE TEST:", dataset.upper())
    print("=" * 70)

    run_one("swininsseg", dataset, smoke=True)

print("\n✅ Both smoke tests passed")


SMOKE TEST: SSDD
Running: swininsseg ssdd | smoke: True
ClearML Task: created new task id=5b4abe89da0a4b088c2b01fc57a8ae04
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/5b4abe89da0a4b088c2b01fc57a8ae04/output/log
Hashing ssdd source images before grouped splitting
Excluded 12 pool image(s) overlapping the test set
Fingerprinting ssdd/train: 8 images
Fingerprinting ssdd/val: 4 images
Fingerprinting ssdd/test: 4 images
{'train': 8, 'val': 4, 'test': 4}
09/05 09:31:09 - mmengine - INFO - Loads checkpoint by http backend from path: https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth
Downloading: "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth" to /root/.cache/torch/hub/checkpoints/swin_tiny_patch4_window7_224.pth
09/05 09:31:10 - mmengine - INFO - 
backbone.patch_embed.projection.weight - torch.Size([96, 3, 4, 4]): 
Initialized by user-defined

In [ ]:
for dataset in ["ssdd", "hrsid"]:
    smoke_folder = (
        ROOT
        / "runs"
        / dataset
        / "swininsseg"
        / "study_v9_swininsseg_fresh_smoke"
    )

    marker = smoke_folder / "evaluation_complete.json"

    print(
        dataset,
        "| smoke complete:",
        marker.is_file(),
    )

    assert marker.is_file(), f"Smoke test incomplete: {dataset}"

print("Safe to start full training")

ssdd | smoke complete: True
hrsid | smoke complete: True
Safe to start full training


In [ ]:
for dataset in ["ssdd", "hrsid"]:
    print("\nSmoke test:", dataset.upper())
    run_one("swininsseg", dataset, smoke=True)

print("\n✅ Both smoke tests completed")


Smoke test: SSDD
Running: swininsseg ssdd | smoke: True
ClearML Task: created new task id=a322858bb4004bd6bd4b41f15f09e47d
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/a322858bb4004bd6bd4b41f15f09e47d/output/log
Hashing ssdd source images before grouped splitting
Excluded 12 pool image(s) overlapping the test set
Fingerprinting ssdd/train: 8 images
Fingerprinting ssdd/val: 4 images
Fingerprinting ssdd/test: 4 images
{'train': 8, 'val': 4, 'test': 4}
09/05 10:26:04 - mmengine - INFO - Loads checkpoint by http backend from path: https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth
09/05 10:26:04 - mmengine - INFO - 
backbone.patch_embed.projection.weight - torch.Size([96, 3, 4, 4]): 
Initialized by user-defined `init_weights` in SwinTransformer  
 
09/05 10:26:04 - mmengine - INFO - 
backbone.patch_embed.projection.bias - torch.Size([96]): 
Initialized by user-defined `init_weights` in SwinTran

In [ ]:
for dataset in settings['datasets']:
    run_one('solov2_gala', dataset, smoke=True)


ClearML Task: created new task id=ac2dc84afeb04f5ab9fd7b021f823bb5
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/ac2dc84afeb04f5ab9fd7b021f823bb5/output/log
Hashing ssdd source images before grouped splitting
Excluded 12 pool image(s) overlapping the test set
Fingerprinting ssdd/train: 8 images
Fingerprinting ssdd/val: 4 images
Fingerprinting ssdd/test: 4 images
{'train': 8, 'val': 4, 'test': 4}
09/06 12:53:43 - mmengine - INFO - load model from: torchvision://resnet50
09/06 12:53:43 - mmengine - INFO - Loads checkpoint by torchvision backend from path: torchvision://resnet50
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
09/06 12:53:44 - mmengine - WARNING - The model and loaded state dict do not match exactly

unexpected key in source state_dict: fc.weight, fc.bias

09/06 12:53:44 - mmengine - INFO - 
backbone.conv1.weight - torch.Size([64, 3, 7, 7]): 
Pr

In [ ]:
import json

BASE_CONFIG = ROOT / "configs" / "study.json"
CONFIG = ROOT / "configs" / "mmdetection_v10_stable.json"

settings = json.loads(BASE_CONFIG.read_text())

settings["experiment"] = "study_v10_swininsseg_stable"
settings["models"] = ["swininsseg"]
settings["datasets"] = ["ssdd", "hrsid"]

settings["max_epochs"] = 500
settings["patience"] = 20
settings["seed"] = 42

settings["batch_size"] = 1
settings["workers"] = 0

# Stability corrections
settings["amp"] = False
settings["learning_rate"] = 0.00002

settings["clearml"] = True

CONFIG.write_text(json.dumps(settings, indent=2))

print("Experiment:", settings["experiment"])
print("Learning rate:", settings["learning_rate"])
print("Mixed precision:", settings["amp"])
print("Batch size:", settings["batch_size"])
print("Workers:", settings["workers"])
print("ClearML:", settings["clearml"])

Experiment: study_v10_swininsseg_stable
Learning rate: 2e-05
Mixed precision: False
Batch size: 1
Workers: 0
ClearML: True


In [ ]:
import json

CONFIG = ROOT / "configs" / "mmdetection_v10_stable.json"
settings = json.loads(CONFIG.read_text())

assert settings["experiment"] == "study_v10_swininsseg_stable"
assert settings["amp"] is False
assert settings["learning_rate"] == 0.00002
assert settings["batch_size"] == 1
assert settings["workers"] == 0

print("Experiment:", settings["experiment"])
print("Configuration loaded correctly")

Experiment: study_v10_swininsseg_stable
Configuration loaded correctly


In [ ]:
import json

BASE_CONFIG = ROOT / "configs" / "study.json"
CONFIG = ROOT / "configs" / "solov2_gala_v11.json"

settings = json.loads(BASE_CONFIG.read_text())

settings["experiment"] = "study_v11_solov2_gala"
settings["models"] = ["solov2_gala"]
settings["datasets"] = ["ssdd", "hrsid"]

settings["max_epochs"] = 500
settings["patience"] = 20
settings["seed"] = 42

settings["batch_size"] = 1
settings["workers"] = 0

# Stable MMDetection training
settings["amp"] = False
settings["learning_rate"] = 0.0001

settings["clearml"] = True

CONFIG.write_text(json.dumps(settings, indent=2))

print("Experiment:", settings["experiment"])
print("Models:", settings["models"])
print("Datasets:", settings["datasets"])
print("Learning rate:", settings["learning_rate"])
print("Mixed precision:", settings["amp"])
print("Batch size:", settings["batch_size"])
print("Workers:", settings["workers"])
print("ClearML:", settings["clearml"])

Experiment: study_v11_solov2_gala
Models: ['solov2_gala']
Datasets: ['ssdd', 'hrsid']
Learning rate: 0.0001
Mixed precision: False
Batch size: 1
Workers: 0
ClearML: True


In [ ]:
def run_one(model, dataset, smoke=True):
    command = [
        str(PYTHON),
        "-u",
        "-m",
        "sarstudy.runner",
        "--config",
        str(CONFIG),
        "--model",
        model,
        "--dataset",
        dataset,
    ]

    if smoke:
        command.append("--smoke")

    environment = os.environ.copy()
    environment["MPLBACKEND"] = "Agg"

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=environment,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"{model}/{dataset} failed with exit code {return_code}"
        )

    print(f"\n✅ Completed: {model}/{dataset}")

In [ ]:
import os
import subprocess

def run_one(model, dataset, smoke=True):
    command = [
        str(PYTHON),
        "-u",
        "-m",
        "sarstudy.runner",
        "--config",
        str(CONFIG),
        "--model",
        model,
        "--dataset",
        dataset,
    ]

    if smoke:
        command.append("--smoke")

    environment = os.environ.copy()
    environment["MPLBACKEND"] = "Agg"

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=environment,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"{model}/{dataset} failed with exit code {return_code}"
        )

    print(f"\n✅ Completed: {model}/{dataset}")

In [8]:
import os
import subprocess

def run_one(model, dataset, smoke=True):
    command = [
        str(PYTHON),
        "-u",
        "-m",
        "sarstudy.runner",
        "--config",
        str(CONFIG),
        "--model",
        model,
        "--dataset",
        dataset,
    ]

    if smoke:
        command.append("--smoke")

    environment = os.environ.copy()
    environment["MPLBACKEND"] = "Agg"

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=environment,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"{model}/{dataset} failed with exit code {return_code}"
        )

    print(f"\n✅ Completed: {model}/{dataset}")

In [9]:
for dataset in ["ssdd", "hrsid"]:
    folder = (
        ROOT
        / "runs"
        / dataset
        / "solov2_gala"
        / "study_v11_solov2_gala"
    )

    history_file = folder / "history.json"

    print("\n", dataset.upper())
    print("Training complete:", (folder / "training_complete.json").is_file())
    print("Evaluation complete:", (folder / "evaluation_complete.json").is_file())
    print("Last checkpoint:", (folder / "checkpoints" / "last.pt").is_file())

    if history_file.is_file():
        history = json.loads(history_file.read_text())
        print("Last saved epoch:", history[-1]["epoch"])


 SSDD
Training complete: True
Evaluation complete: True
Last checkpoint: True
Last saved epoch: 21

 HRSID
Training complete: False
Evaluation complete: False
Last checkpoint: True
Last saved epoch: 5


In [ ]:
SSDD_FOLDER = (
    ROOT
    / "runs"
    / "ssdd"
    / "swininsseg"
    / "study_v10_swininsseg_stable"
)

print(
    "Training complete:",
    (SSDD_FOLDER / "training_complete.json").is_file(),
)
print(
    "Best checkpoint:",
    (SSDD_FOLDER / "checkpoints" / "best.pt").is_file(),
)
print(
    "Final evaluation complete:",
    (SSDD_FOLDER / "evaluation_complete.json").is_file(),
)

Training complete: True
Best checkpoint: True
Final evaluation complete: False


## Long runs — only after checking smoke results

Enable this deliberately. Each model/dataset has its own history and checkpoint folder. If Colab disconnects, repeat setup and rerun the same cell; saved state is resumed. Do not run two copies of the same experiment simultaneously.

In [ ]:
APPROVE_LONG_RUNS = True  # Change to True only after checking smoke logs and curves.


In [ ]:
for dataset in ["ssdd", "hrsid"]:
    print("\n" + "=" * 70)
    print("FULL TRAINING:", dataset.upper())
    print("=" * 70)

    run_one(
        model="swininsseg",
        dataset=dataset,
        smoke=False,
    )

print("\n✅ SwinInsSeg completed on SSDD and HRSID")


FULL TRAINING: SSDD
Running: swininsseg ssdd | smoke: False
ClearML Task: created new task id=c686a225ab2d430db5096be4c85b1810
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/c686a225ab2d430db5096be4c85b1810/output/log
Hashing ssdd source images before grouped splitting
Excluded 12 pool image(s) overlapping the test set
Fingerprinting ssdd/train: 733 images
Fingerprinting ssdd/val: 183 images
Fingerprinting ssdd/test: 232 images
{'train': 733, 'val': 183, 'test': 232}
09/05 10:32:39 - mmengine - INFO - Loads checkpoint by http backend from path: https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth
09/05 10:32:39 - mmengine - INFO - 
backbone.patch_embed.projection.weight - torch.Size([96, 3, 4, 4]): 
Initialized by user-defined `init_weights` in SwinTransformer  
 
09/05 10:32:39 - mmengine - INFO - 
backbone.patch_embed.projection.bias - torch.Size([96]): 
Initialized by user-defined `init_weig

In [ ]:
for dataset in ["ssdd", "hrsid"]:
    print("\n" + "=" * 70)
    print("CONTINUING:", dataset.upper())
    print("=" * 70)

    run_one("swininsseg", dataset, smoke=False)

print("\n✅ SSDD and HRSID are fully complete")


CONTINUING: SSDD
ClearML Task: created new task id=9ace6e88abe34c0a924dc265ab7cce02
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/9ace6e88abe34c0a924dc265ab7cce02/output/log
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start
{'train': 733, 'val': 183, 'test': 232}
09/05 14:50:39 - mmengine - INFO - Loads checkpoint by http backend from path: https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth
Downloading: "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth" to /root/.cache/torch/hub/checkpoints/swin_tiny_patch4_window7_224.pth
09/05 14:50:43 - mmengine - INFO - 
backbone.patch_embed.projection.weight - torch.Size([96, 3, 4, 4]): 
Initialized by user-defined `init_weights` in SwinTransformer  
 
09/05 14:50:43 - mmengine - INFO - 
backbone.patch_embed.projection.bias - torch.Size([96]): 
Init

In [ ]:
for dataset in ["ssdd", "hrsid"]:
    print("\n" + "=" * 70)
    print("CONTINUING:", dataset.upper())
    print("=" * 70)

    run_one("swininsseg", dataset, smoke=False)

print("\n✅ Both datasets completed")


CONTINUING: SSDD


NameError: name 'run_one' is not defined

In [6]:

import json

CONFIG = ROOT / "configs" / "solov2_gala_v11.json"
settings = json.loads(CONFIG.read_text())

assert settings["experiment"] == "study_v11_solov2_gala"
assert settings["models"] == ["solov2_gala"]

print("Experiment:", settings["experiment"])
print("Datasets:", settings["datasets"])
print("ClearML:", settings["clearml"])

Experiment: study_v11_solov2_gala
Datasets: ['ssdd', 'hrsid']
ClearML: True


In [ ]:
import time

def run_with_resume(model, dataset, max_restarts=20):
    for attempt in range(1, max_restarts + 1):
        try:
            print(f"\n{model}/{dataset} — attempt {attempt}")
            run_one(model, dataset, smoke=False)
            print(f"\n✅ Completed: {model}/{dataset}")
            return

        except RuntimeError as error:
            if "exit code -9" not in str(error):
                raise

            print(
                f"\n⚠️ Process was killed during {model}/{dataset}."
                "\nThe saved checkpoint will be resumed automatically."
            )
            time.sleep(10)

    raise RuntimeError(
        f"{model}/{dataset} was interrupted {max_restarts} times"
    )


assert APPROVE_LONG_RUNS, "Inspect smoke results first"

for dataset in ["ssdd", "hrsid"]:
    run_with_resume("swininsseg", dataset)


swininsseg/ssdd — attempt 1
ClearML Task: created new task id=a1e1cce56c4944aa925d805b193c888a
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/a1e1cce56c4944aa925d805b193c888a/output/log
{'train': 733, 'val': 183, 'test': 232}
09/05 08:54:56 - mmengine - INFO - Loads checkpoint by http backend from path: https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth
09/05 08:54:56 - mmengine - INFO - 
backbone.patch_embed.projection.weight - torch.Size([96, 3, 4, 4]): 
Initialized by user-defined `init_weights` in SwinTransformer  
 
09/05 08:54:56 - mmengine - INFO - 
backbone.patch_embed.projection.bias - torch.Size([96]): 
Initialized by user-defined `init_weights` in SwinTransformer  
 
09/05 08:54:56 - mmengine - INFO - 
backbone.patch_embed.norm.weight - torch.Size([96]): 
Initialized by user-defined `init_weights` in SwinTransformer  
 
09/05 08:54:56 - mmengine - INFO - 
backbone.patch_embed.norm.

In [ ]:
assert APPROVE_LONG_RUNS, 'Inspect smoke results first'
for dataset in settings['datasets']:
    run_one('solov2_gala', dataset, smoke=False)


ClearML Task: created new task id=3afb6170fefb496993ab7c59043bc6f6
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/3afb6170fefb496993ab7c59043bc6f6/output/log
Hashing ssdd source images before grouped splitting
Excluded 12 pool image(s) overlapping the test set
Fingerprinting ssdd/train: 733 images
Fingerprinting ssdd/val: 183 images
Fingerprinting ssdd/test: 232 images
{'train': 733, 'val': 183, 'test': 232}
09/06 13:14:08 - mmengine - INFO - load model from: torchvision://resnet50
09/06 13:14:08 - mmengine - INFO - Loads checkpoint by torchvision backend from path: torchvision://resnet50
09/06 13:14:08 - mmengine - WARNING - The model and loaded state dict do not match exactly

unexpected key in source state_dict: fc.weight, fc.bias

09/06 13:14:08 - mmengine - INFO - 
backbone.conv1.weight - torch.Size([64, 3, 7, 7]): 
PretrainedInit: load from torchvision://resnet50 
 
09/06 13:14:08 - mmengine - INFO - 
backbone.bn1.weight - torch.Size([6

In [10]:
for dataset in ["ssdd", "hrsid"]:
    print("\n" + "=" * 70)
    print("CONTINUING SOLOv2-GALA:", dataset.upper())
    print("=" * 70)

    run_one("solov2_gala", dataset, smoke=False)

print("\n✅ SOLOv2-GALA completed on both datasets")


CONTINUING SOLOv2-GALA: SSDD
ClearML Task: created new task id=a7e6e1db4b1b4de291254c71f75e9206
ClearML results page: https://app.clear.ml/projects/9ab57627625e4512b3152bb8a6a3224c/tasks/a7e6e1db4b1b4de291254c71f75e9206/output/log
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start
{'train': 733, 'val': 183, 'test': 232}
09/06 15:50:15 - mmengine - INFO - load model from: torchvision://resnet50
09/06 15:50:15 - mmengine - INFO - Loads checkpoint by torchvision backend from path: torchvision://resnet50
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
09/06 15:50:16 - mmengine - WARNING - The model and loaded state dict do not match exactly

unexpected key in source state_dict: fc.weight, fc.bias

09/06 15:50:16 - mmengine - INFO - 
backbone.conv1.weight - torch.Size([64, 3, 7, 7]): 
PretrainedInit: load from torchvision://resnet50 
 
09/06 15:50:16 - m

## Check durable outputs

In [ ]:
for dataset in settings['datasets']:
    for model in settings['models']:
        folder = ROOT/'runs'/dataset/model/settings['experiment']
        print(dataset, model, 'training:', (folder/'training_complete.json').is_file(),
              'evaluation:', (folder/'evaluation_complete.json').is_file())
